### Fashion-MNIST Classifier using LeNet-5

In [1]:
import torch
import torchvision
from torch import nn
from torch.utils import data
from torchvision import transforms
import matplotlib.pyplot as plt

In [2]:
net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(), 
    nn.Linear(16 * 5 * 5, 120), nn.Sigmoid(),
    nn.Linear(120, 84), nn.Sigmoid(),
    nn.Linear(84, 10)
)

shape test:

In [4]:
X = torch.randn((1, 1, 28, 28), dtype=torch.float32, requires_grad=False)
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape: \t',X.shape)

Conv2d output shape: 	 torch.Size([1, 6, 28, 28])
Sigmoid output shape: 	 torch.Size([1, 6, 28, 28])
AvgPool2d output shape: 	 torch.Size([1, 6, 14, 14])
Conv2d output shape: 	 torch.Size([1, 16, 10, 10])
Sigmoid output shape: 	 torch.Size([1, 16, 10, 10])
AvgPool2d output shape: 	 torch.Size([1, 16, 5, 5])
Flatten output shape: 	 torch.Size([1, 400])
Linear output shape: 	 torch.Size([1, 120])
Sigmoid output shape: 	 torch.Size([1, 120])
Linear output shape: 	 torch.Size([1, 84])
Sigmoid output shape: 	 torch.Size([1, 84])
Linear output shape: 	 torch.Size([1, 10])


In [5]:
trans = transforms.ToTensor()
mnist_data = torchvision.datasets.FashionMNIST(
    root="./data", train=True, transform=trans, download=True)
mnist_test = torchvision.datasets.FashionMNIST(
    root="./data", train=False, transform=trans, download=True)
len(mnist_data), len(mnist_test)

(60000, 10000)

In [7]:
batch_size = 256
dataloader = data.DataLoader(mnist_data, batch_size=batch_size, shuffle=True)
testloader = data.DataLoader(mnist_test, batch_size=batch_size, shuffle=True)
feature, label = next(iter(dataloader))
feature.shape, label.shape

(torch.Size([256, 1, 28, 28]), torch.Size([256]))

In [8]:
class Accumulator:
    """Accumulator class for accumulating multiple scalar values."""

    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, key):
        return self.data[key]


def accuracy(y_hat, y):
    """Calculate the number of correct predictions.

    Args:
        y_hat: Model predictions with shape (batch_size, num_classes)
        y: Ground truth labels with shape (batch_size,)

    Returns:
        float: Number of correct predictions
    """
    y_hat = y_hat.argmax(axis=1)
    cmp = y_hat == y
    return cmp.float().sum().item()


def evaluator(net, test_iter, device=None):
    """Evaluate model accuracy on the test set.

    Args:
        net: Neural network model
        test_iter: Test data iterator

    Returns:
        float: Accuracy on the test set
    """
    if isinstance(net, nn.Module):
        net.eval()
        if not device:
            device = device = next(net.parameters()).device
    metric = Accumulator(2)
    with torch.no_grad():
        for X, y in test_iter:
            if isinstance(X, list):
                # BERT微调所需的（之后将介绍）
                X = [x.to(device) for x in X]
            else:
                X = X.to(device)
            y = y.to(device)
            metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

In [9]:
def train_ch6(net, train_iter, test_iter, num_epochs, lr, device):
    """用GPU训练模型(在第六章定义)"""
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    print('training on', device)
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    num_batches = len(train_iter)
    for epoch in range(num_epochs):
        # 训练损失之和，训练准确率之和，样本数
        metric = Accumulator(3)
        net.train()
        for i, (X, y) in enumerate(train_iter):
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()
            with torch.no_grad():
                metric.add(l * X.shape[0], accuracy(y_hat, y), X.shape[0])
            train_l = metric[0] / metric[2]
            train_acc = metric[1] / metric[2]
        test_acc = evaluator(net, test_iter)
    print(f'loss {train_l:.5f}, train acc {train_acc:.5f}, '
          f'test acc {test_acc:.5f}')
    return train_acc, test_acc

In [ ]:
lr, num_epochs = 0.9, 10
train_ch6(net, dataloader, testloader, num_epochs, lr, 'gpu')